# 02 - RQ1: Baseline and Imbalance-Corrected Models (incl. TabNet)

**Notebook version:** v5 -- 2026-07-29

- Fit logistic regression, Random Forest, XGBoost, and TabNet with stratified k-fold CV
- Apply class weighting and SMOTE as a robustness check
- Compare all four models on AUC-ROC, AUC-PR, and Balanced Error Rate (BER)
- Report whether TabNet offers a measurable edge over tree ensembles at this sample size


In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# Also pulls latest changes and prints the commit hash, so you can confirm at a
# glance (against GitHub's commit history) that you're looking at the current version.
import os
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
    else:
        os.chdir(f"/content/{REPO_NAME}")
        !git pull
        os.chdir("/content")
    os.chdir(f"/content/{REPO_NAME}/notebooks")
    !pip install -q -r ../requirements.txt

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)


In [ ]:
import numpy as np
import pandas as pd

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from metrics import summarize, balanced_error_rate, mcc, auc_roc

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))
y = (y == 1).astype(int)  # 1 = fail (positive class), 0 = pass

print(f"Screened feature count: {X.shape[1]}")
print(f"Class balance: {y.value_counts().to_dict()} (fail rate = {y.mean():.2%})")

## Models and imbalance correction

Four model families, each imbalance-corrected in the way appropriate to its
own training procedure (class weighting, not resampling, is the primary
correction here -- SMOTE is run separately below as the robustness check
described in the synopsis, not the default):
- Logistic Regression (`class_weight='balanced'`)
- Random Forest (`class_weight='balanced'`)
- XGBoost (`scale_pos_weight` set to the pass:fail ratio)
- TabNet (per-sample weights passed at `.fit()`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

pos_weight = (y == 0).sum() / (y == 1).sum()

def make_models():
    return {
        "LogisticRegression": LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=4, scale_pos_weight=pos_weight,
            eval_metric="logloss", random_state=42,
        ),
    }

## Stratified k-fold cross-validation (out-of-fold predictions)

5-fold stratified CV, collecting out-of-fold predicted probabilities for
each sklearn/XGBoost model so every wafer gets exactly one held-out
prediction -- avoids the leakage risk of fitting and scoring on the same
fold, and gives a fair AUC-ROC/BER/MCC comparison across models.

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_probs = {name: np.zeros(len(y)) for name in make_models()}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for name, model in make_models().items():
        model.fit(X_train, y_train)
        oof_probs[name][test_idx] = model.predict_proba(X_test)[:, 1]

    print(f"Fold {fold + 1}/{N_FOLDS} complete")

## TabNet (same CV scheme, separate cell -- slower to train)

TabNet needs numpy arrays (not DataFrames) and its own per-sample weight
argument rather than a `class_weight` constructor param; kept in its own
cell since it trains meaningfully slower than the sklearn/XGBoost models
above and benefits from being re-run independently while iterating.

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier

X_arr = X.values.astype("float32")
y_arr = y.values

oof_probs["TabNet"] = np.zeros(len(y))
sample_weight = np.where(y_arr == 1, pos_weight, 1.0)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_arr, y_arr)):
    tabnet = TabNetClassifier(seed=42, verbose=0)
    tabnet.fit(
        X_arr[train_idx], y_arr[train_idx],
        weights=1,  # 1 = auto-balance by inverse class frequency
        max_epochs=60, patience=10, batch_size=128,
    )
    oof_probs["TabNet"][test_idx] = tabnet.predict_proba(X_arr[test_idx])[:, 1]
    print(f"TabNet fold {fold + 1}/{N_FOLDS} complete")

## RQ1 results: model comparison

Threshold for BER/MCC (binary predictions) is 0.5 on the OOF probability;
revisit this if a model's OOF probabilities are poorly calibrated --
AUC-ROC and AUC-PR below are threshold-independent and are the primary
comparison metrics for that reason.

In [ ]:
from sklearn.metrics import average_precision_score

rows = []
for name, probs in oof_probs.items():
    preds = (probs >= 0.5).astype(int)
    m = summarize(y, preds, probs)
    m["model"] = name
    m["auc_pr"] = average_precision_score(y, probs)
    rows.append(m)

results_df = pd.DataFrame(rows).set_index("model")[
    ["auc_roc", "auc_pr", "ber", "mcc", "precision", "recall", "f1"]
].sort_values("auc_roc", ascending=False)

results_df.round(4)

## Visual comparison and the RQ1 answer

Per the synopsis: does TabNet offer any measurable advantage over the
tree ensembles here? Compare its AUC-ROC/BER/MCC against XGBoost and
Random Forest directly below rather than assuming either outcome.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results_df["auc_roc"].sort_values().plot.barh(ax=axes[0], title="AUC-ROC by model")
results_df["ber"].sort_values(ascending=False).plot.barh(ax=axes[1], title="Balanced Error Rate (lower is better)")
plt.tight_layout()
plt.savefig("../docs/rq1_model_comparison.png", dpi=150)
plt.show()

best_model = results_df["auc_roc"].idxmax()
tabnet_rank = results_df.index.get_loc("TabNet") + 1
print(f"Best model by AUC-ROC: {best_model}")
print(f"TabNet ranks #{tabnet_rank} of {len(results_df)} by AUC-ROC")

## Save results for downstream notebooks

Without this step, `03`-`06` would each silently re-derive their own
assumptions instead of using what RQ1 actually found. Two things are saved,
not one, because the synopsis's own methodology needs both: RQ2's SHAP
analysis is scoped to "the best-performing **tree** ensemble" specifically
(TreeExplainer only supports tree models), while RQ1's own question --
does TabNet beat tree ensembles -- needs the true overall winner. If those
two happen to be the same model, both artifacts just point to it; if not,
downstream notebooks get the correct one for what they're actually doing.

In [ ]:
from artifacts import save_model, save_json, tabnet_save_path

best_overall = results_df["auc_roc"].idxmax()
tree_only = results_df.loc[results_df.index.intersection(["RandomForest", "XGBoost"])]
best_tree = tree_only["auc_roc"].idxmax()

print(f"Best model overall (by AUC-ROC): {best_overall}")
print(f"Best TREE-ensemble model (by AUC-ROC): {best_tree}")
if best_overall != best_tree:
    print(
        f"Note: {best_overall} beat the tree ensembles overall, but RQ2/RQ3's "
        f"SHAP analysis will still use {best_tree} (the best tree model) per "
        f"the synopsis's stated TreeExplainer-based methodology."
    )

def refit_full(name):
    if name == "LogisticRegression":
        m = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
        m.fit(X, y)
        return m
    if name == "RandomForest":
        m = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
        m.fit(X, y)
        return m
    if name == "XGBoost":
        m = XGBClassifier(
            n_estimators=200, max_depth=4, scale_pos_weight=pos_weight,
            eval_metric="logloss", random_state=42,
        )
        m.fit(X, y)
        return m
    raise ValueError(f"Unexpected model name: {name}")

# Always save the best tree model -- this is what 03/04 will load.
best_tree_model = refit_full(best_tree)
save_model("rq1_best_tree", best_tree_model)

# Separately save the overall winner, refitting on full data only if it
# differs from the tree model already fit above (avoid training TabNet twice).
if best_overall == "TabNet":
    tabnet_full = TabNetClassifier(seed=42, verbose=0)
    tabnet_full.fit(X_arr, y_arr, weights=1, max_epochs=60, patience=10, batch_size=128)
    tabnet_full.save_model(tabnet_save_path("rq1_best_overall_tabnet"))
elif best_overall != best_tree:
    overall_model = refit_full(best_overall)
    save_model("rq1_best_overall", overall_model)

save_json("rq1_summary", {
    "best_overall_model_name": best_overall,
    "best_tree_model_name": best_tree,
    "screened_feature_count": int(X.shape[1]),
    "feature_names": list(X.columns),
    "results": results_df.reset_index().to_dict(orient="records"),
})

print("\nSaved to data/artifacts/: rq1_best_tree.joblib, rq1_summary.json"
      + (", rq1_best_overall_tabnet.zip" if best_overall == "TabNet"
         else ", rq1_best_overall.joblib" if best_overall != best_tree else ""))

## Next steps

- Report `results_df` and the AUC-ROC/BER comparison plot in the capstone write-up as the RQ1 deliverable
- Carry the best tree-ensemble model forward into `03_shap_analysis_rq2.ipynb`
- Carry `oof_probs["TabNet"]` and a control MLP forward into `05_attention_comparison_rq4.ipynb`

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "02_modeling_rq1"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed.
    from google.colab import _message
    ipynb_content = _message.blocking_request('get_ipynb', timeout_sec=30)['ipynb']
    with open(export_path, 'w') as f:
        json.dump(ipynb_content, f)

html_output = f"{NOTEBOOK_NAME}.html"
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print(f"Exported to {html_output}")

if IN_COLAB and result.returncode == 0:
    from google.colab import files
    files.download(html_output)
